# Data Sharing: Atlas Charts

---

This notebook will prepare a list of charts for various brain atlases. (Showcasing that a single pretrained SNM can be used for various spatial queries)


### Package imports and basic functions

---

In [1]:
import os
import gc
import sys
import glob
import shutil
import json
import random
import datetime
import importlib
import itertools
import numpy as np
from scipy import spatial
import scipy.sparse as sparse
import scipy.stats as stats
import pandas as pd
import nibabel as nib
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
from tqdm.auto import tqdm
from urllib.parse import urlparse
import requests
import zipfile
from pathlib import Path
import polars as pl
import colorcet as cc
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.image as mpimg
import joblib
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.gridspec import GridSpec


In [2]:
%load_ext autoreload
%autoreload 2

# Path to add to src folder (to use local version of spectranorm)
sys.path.append(os.path.abspath("/mountpoint/code/projects/spectranorm/package/spectranorm/src/"))

from spectranorm import snm


In [3]:
class MyNumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        else:
            return super(MyEncoder, self).default(obj)


def ensure_dir(file_name):
    os.makedirs(os.path.dirname(file_name), exist_ok=True)
    return file_name


def list_dirs(path=os.getcwd()):
    files = glob.glob(os.path.join(path, '*'))
    files = [x for x in files if os.path.isdir(x)]
    return files


def file_exists(file_name, path_name=os.getcwd()):
    return os.path.isfile(os.path.join(path_name, file_name))


def write_json(json_obj, file_path):
    with open(file_path, 'w') as outfile:
        json.dump(json_obj, outfile, sort_keys=True, indent=4,
                  cls=MyNumpyEncoder)
    return json_obj


def load_json(file_path):
    with open(file_path, 'r') as infile:
        return json.load(infile)


def write_np(np_obj, file_path):
    with open(file_path, 'wb') as outfile:
        np.save(outfile, np_obj)


In [4]:
from PIL import Image as PILImage
from IPython.display import display
import warnings

warnings.simplefilter('ignore', PILImage.DecompressionBombWarning)

def show_image(image_path, width=1000, resample=PILImage.LANCZOS):
    """
    Display an image in Jupyter at a fixed width while preserving aspect ratio.
    The image is resampled (not embedded full-size) to reduce notebook memory.
    """
    img = PILImage.open(image_path).copy()
    w, h = img.size
    new_h = int(h * width / w)
    img = img.resize((width, new_h), resample)
    display(img)


In [5]:
# use tex for plotting
plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Computer Modern']


### Brain visualization functions

---

Brain visualization scripts (utilizing [Cerebro Brain Viewer](https://cerebro-viewer.readthedocs.io/en/latest/))

In [6]:
# Cerebro brain viewer used for visualization
from cerebro import cerebro_brain_utils as cbu
from cerebro import cerebro_brain_viewer as cbv


In [7]:
# basic parameters
surface = 'pial'
expand = 0

# load an example dscalar
dscalar_file = cbu.cifti_template_file
dscalar = nib.load(dscalar_file)

brain_models = [x for x in dscalar.header.get_index_map(1).brain_models]

# load surfaces for visualization
left_surface_file, right_surface_file = cbu.get_left_and_right_GIFTI_template_surface(surface)
left_surface = nib.load(left_surface_file)
right_surface = nib.load(right_surface_file)

# extract surface information
lx, ly, lz = left_surface.darrays[0].data.T
lt = left_surface.darrays[1].data
rx, ry, rz = right_surface.darrays[0].data.T
rt = right_surface.darrays[1].data

# combine into a complete brain
lrx = np.concatenate([lx - expand, rx + expand])
lry = np.concatenate([ly, ry])
lrz = np.concatenate([lz, rz])
lrt = np.concatenate([lt, (rt + lx.shape[0])])

lxyz = left_surface.darrays[0].data
rxyz = right_surface.darrays[0].data
lrxyz = np.array([lrx, lry, lrz]).T

# create a mapping between surface and cifti vertices
left_cortical_surface_model, right_cortical_surface_model = brain_models[0], brain_models[1]
cifti_to_surface = {}
surface_to_cifti = {}
for (i, x) in enumerate(left_cortical_surface_model.vertex_indices):
    cifti_to_surface[i] = x
    surface_to_cifti[x] = i
for (i, x) in enumerate(right_cortical_surface_model.vertex_indices):
    cifti_to_surface[i + right_cortical_surface_model.index_offset] = x + rx.shape[0]
    surface_to_cifti[x + rx.shape[0]] = i + right_cortical_surface_model.index_offset

# construct data over surface
surface_mask = list(surface_to_cifti.keys())


In [8]:
# arbitrary colormap
mycm = LinearSegmentedColormap.from_list(
    'my_gradient',
    (
        (0.0, (0.1, 0.1, 1.,)),
        # (0.4999, (0.9, 0.9, 1.,)),
        (0.25, (0.1, 1., 1.,)),
        # (0.4999, (0.9, 1., 1.,)),
        (0.5, (1., 1., 1.,)),
        # (0.5001, (1., 1., 0.9,)),
        (0.75, (1., 1., 0.1,)),
        # (0.5001, (1., 0.9, 0.9,)),
        (1.0, (1., 0.1, 0.1,)),
    )
)

mycm_dark = LinearSegmentedColormap.from_list(
    'my_gradient',
    (
        (0.0, (0.05, 1., 1.,)),
        (0.35, (0.05, 0.05, 1.,)),
        (0.5, (0.2, 0.05, 0.05,)),
        (0.65, (1., 0.05, 0.05,)),
        (1.0, (1., 1., 0.05,)),
    )
)


In [9]:
# Brain visualizations with Cerebro

# ignore warning when loading cifti
nib.imageglobals.logger.setLevel(40)

# suppress Mesa/GL warnings
os.environ["LIBGL_DEBUG"] = "quiet"

from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.gridspec import GridSpec

def plot_single_view_with_cerebro(
        dscalar_data, ax, colormap=plt.cm.coolwarm, clims=None, vlims=None, exclusion_color=(1.,1.,1.,0),
        view="L", show_colorbar=False, colorbar_format=None, cifti_left_right_seperation=0,
        surface = 'midthickness', spheres=None):
    try:
        
        my_brain_viewer = cbv.Cerebro_brain_viewer(offscreen=True, background_color=(1.,1.,1.,0), null_color=(1., 1., 1., 0.0), no_color=(0.7, 0.7, 0.7, 1.))

        surface_model = my_brain_viewer.load_template_GIFTI_cortical_surface_models(surface)

        cifti_space = my_brain_viewer.visualize_cifti_space(
            volumetric_structures='none', cifti_left_right_seperation=cifti_left_right_seperation,
        )

        dscalar_layer = my_brain_viewer.add_cifti_dscalar_layer(
            dscalar_data=dscalar_data,
            colormap=colormap,
            clims=clims,
            vlims=vlims,
            exclusion_color=exclusion_color,
            opacity=0.95)
        
        if spheres is not None:
            my_brain_viewer.visualize_spheres(
                coordinates=spheres[0] + (spheres[2] @ np.array([[cifti_left_right_seperation/2, 0, 0]])),
                radii=1*np.ones(spheres[0].shape),
                color=spheres[1],
            )

        ax.axis('off')
        camconf = my_brain_viewer._view_to_camera_config(view)
        # camconf = my_brain_viewer.zoom_camera_to_content(camconf)
        my_brain_viewer.viewer.change_view(**camconf)
        my_brain_viewer.offscreen_draw_to_matplotlib_axes(ax)

    finally:
        my_brain_viewer.viewer.window.destroy()

def plot_left_right_surface_with_cerebro(dscalar_data, fig, ax, colormap=plt.cm.coolwarm, clims=None, show_colorbar=False, spheres=None, colorbar_format=None, **kwargs):
    # Hide the parent axis
    ax.set_visible(False)

    # Create a 2x2 GridSpec within the axis using inset_axes
    gs = ax.inset_axes([0, 0, 1, 1], transform=ax.transAxes)
    sub_gs = GridSpec(2, 2, gs, hspace=0., wspace=0.)

    # Create a 4x4 grid
    ax_tl = fig.add_subplot(sub_gs[0, 0])
    ax_tr = fig.add_subplot(sub_gs[0, 1])
    ax_bl = fig.add_subplot(sub_gs[1, 0])
    ax_br = fig.add_subplot(sub_gs[1, 1])
    
    # separate data to left and right
    dscalar_data_left = dscalar_data.copy()
    dscalar_data_left[left_cortical_surface_model.index_count:] = np.nan
    dscalar_data_right = dscalar_data.copy()
    dscalar_data_right[:left_cortical_surface_model.index_count] = np.nan
    
    # separate spheres to left and right
    if spheres is None:
        left_spheres = None
        right_spheres = None
    else:
        left_spheres_mask = (spheres[2][:,0] == -1)
        left_spheres = [spheres[0][left_spheres_mask], spheres[1], spheres[2][left_spheres_mask]]
        right_spheres_mask = (spheres[2][:,0] == 1)
        right_spheres = [spheres[0][right_spheres_mask], spheres[1], spheres[2][right_spheres_mask]]
    
    # Lateral left view
    plot_single_view_with_cerebro(dscalar_data_left, ax_tl, colormap=colormap, clims=clims,
                                  view=((-420, 0, 0), None, None, None), spheres=left_spheres, **kwargs)
    # Medial left view
    plot_single_view_with_cerebro(dscalar_data_left, ax_bl, colormap=colormap, clims=clims,
                                  view=((420, 0, 0), None, None, None), spheres=left_spheres, cifti_left_right_seperation=-80, **kwargs)
    
    # Lateral right view
    plot_single_view_with_cerebro(dscalar_data_right, ax_tr, colormap=colormap, clims=clims,
                                  view=((420, 0, 0), None, None, None), spheres=right_spheres, **kwargs)
    # Medial right view
    plot_single_view_with_cerebro(dscalar_data_right, ax_br, colormap=colormap, clims=clims,
                                  view=((-420, 0, 0), None, None, None), spheres=right_spheres, cifti_left_right_seperation=-80, **kwargs)

    if show_colorbar:
        cax = inset_axes(
            ax,
            width="30%",
            height="4%",
            loc="center",
            bbox_to_anchor=(-0., 0.0, 1., 1.),
            bbox_transform=ax.transAxes,
            borderpad=0,
        )
        cb = fig.colorbar(
            mpl.cm.ScalarMappable(
                norm=mpl.colors.Normalize(vmin=clims[0], vmax=clims[1]),
                cmap=colormap
            ),
            cax=cax,
            aspect=10,
            orientation='horizontal',
            format=colorbar_format,
        )
        cb.outline.set_visible(False)
        cb.ax.tick_params(labelsize=12)
        cb.ax.tick_params(length=0)


## Load Spectral Normative Model (SNM-1000)

---

Load the model from 09_01.


In [10]:
%%time
snm_1000 = snm.SpectralNormativeModel.load_model(
    "/mountpoint/code/projects/normative_brain_charts/data/pretrained_models/pretrained_SNM_1000_V1.0/"
)
snm_1000


CPU times: user 4.29 s, sys: 781 ms, total: 5.07 s
Wall time: 5.07 s


SpectralNormativeModel(eigenmode_basis=EigenmodeBasis(n_modes=1000, n_features=59412), base_model=DirectNormativeModel(spec=NormativeModelSpec(variable_of_interest='thickness', covariates=[CovariateSpec(name=age, cov_type=numerical, effect=spline), CovariateSpec(name=sex, cov_type=categorical, hierarchical=False, n_categories=2), CovariateSpec(name=site, cov_type=categorical, hierarchical=True, n_categories=189)], influencing_mean=['age', 'sex', 'site'], influencing_variance=['age', 'sex', 'site'])))

## Automated charting

---

Let's prepare a function that takes a brain parcellation and produces a CSV file of normative trajectories for every parcel.


In [11]:
min_age, max_age = 5, 95
samples_per_year = 1
candidate_ages = np.linspace(min_age, max_age, samples_per_year * (max_age - min_age) + 1)


In [12]:
%%time

spectral_basis_predictions = snm_1000.compute_spectral_predictions(
    test_covariates=pd.DataFrame({"age": candidate_ages,}),
    predict_without=['site', 'sex'],
)
spectral_basis_predictions_male = snm_1000.compute_spectral_predictions(
    test_covariates=pd.DataFrame({"age": candidate_ages, "sex": ["M"]*len(candidate_ages),}),
    predict_without=['site'],
)
spectral_basis_predictions_female = snm_1000.compute_spectral_predictions(
    test_covariates=pd.DataFrame({"age": candidate_ages, "sex": ["F"]*len(candidate_ages),}),
    predict_without=['site'],
)


Computing direct eigenmode estimates:   0%|          | 0/1000 [00:00<?, ?tasks/s]

Computing cross-mode dependence estimates:   0%|          | 0/4889 [00:00<?, ?tasks/s]

Computing direct eigenmode estimates:   0%|          | 0/1000 [00:00<?, ?tasks/s]

Computing cross-mode dependence estimates:   0%|          | 0/4889 [00:00<?, ?tasks/s]

Computing direct eigenmode estimates:   0%|          | 0/1000 [00:00<?, ?tasks/s]

Computing cross-mode dependence estimates:   0%|          | 0/4889 [00:00<?, ?tasks/s]

CPU times: user 8.59 s, sys: 1.14 s, total: 9.73 s
Wall time: 39.9 s


In [13]:
from sklearn.preprocessing import normalize

# a function that turns a positive weighted vector to a weighted average operator (sum of weights will be scaled to 1)
def average_operator(signal):
    return normalize(signal.reshape(1,-1), norm='l1').reshape(-1)


In [14]:
nib.imageglobals.logger.setLevel(40)

def compute_normative_trajectories_for_atlas(atlas_file, output_file, spectral_basis_predictions, lookup=False):
    atlas = nib.load(atlas_file)

    atlas_label_dict = dict(list(atlas.header.get_index_map(0).named_maps)[0].label_table)
    atlas_label_names = {i:atlas_label_dict[i].label for i in atlas_label_dict}
    atlas_label_indices = {atlas_label_dict[i].label:i for i in atlas_label_dict}
    if lookup:
        atlas_data = atlas.get_fdata()[0][surface_mask]
    else:
        atlas_data = atlas.get_fdata()[0][:snm_1000.eigenmode_basis.eigenvectors.shape[0]]

    atlas_queries = {}
    # loop over cortical labels
    for label_index in np.unique(atlas_data):
        if label_index != 0:
            label_name = atlas_label_names[label_index]
            # mean thickness over region
            signal = average_operator(atlas_data == label_index)
            atlas_queries[f'{label_name}'] = signal
    
    atlas_query_names = [x for x in atlas_queries]
    atlas_queries_matrix = np.array([atlas_queries[x] for x in atlas_query_names]).T

    encoded_normative_queries = snm_1000.eigenmode_basis.encode(atlas_queries_matrix.T, n_modes=1000)

    # predict moments
    predicted_moments = snm_1000.predict(
        encoded_query=encoded_normative_queries.T,
        spectral_predictions=spectral_basis_predictions,
        n_modes=1000,
    ).predictions

    # create a dataframe to hold results
    # each row is an age, each column is a region, (two parent columns for mu and sigma)
    predicted_moments_df = pd.DataFrame(
        data=np.concatenate([predicted_moments['mu_estimate'], predicted_moments['std_estimate']], axis=1),
        index=candidate_ages,
        columns=pd.MultiIndex.from_product([['Mean (mm)', 'SD (mm)'], atlas_query_names])
    )
    predicted_moments_df.index.name = 'Age (years)'

    # save to csv
    predicted_moments_df.to_csv(output_file)

    return predicted_moments_df

### DK atlas

---

also known as: aparc

In [15]:
%%time
compute_normative_trajectories_for_atlas(
    atlas_file="/mountpoint/code/projects/scratch_scripts/scripts_for_niousha/cerbero/Sina_New_DK_atlas.dlabel.nii",
    output_file="/mountpoint/code/projects/normative_brain_charts/data/charts/aparc.normative_trajectories.csv",
    spectral_basis_predictions=spectral_basis_predictions,
)
compute_normative_trajectories_for_atlas(
    atlas_file="/mountpoint/code/projects/scratch_scripts/scripts_for_niousha/cerbero/Sina_New_DK_atlas.dlabel.nii",
    output_file="/mountpoint/code/projects/normative_brain_charts/data/charts/aparc.male_normative_trajectories.csv",
    spectral_basis_predictions=spectral_basis_predictions_male,
)
compute_normative_trajectories_for_atlas(
    atlas_file="/mountpoint/code/projects/scratch_scripts/scripts_for_niousha/cerbero/Sina_New_DK_atlas.dlabel.nii",
    output_file="/mountpoint/code/projects/normative_brain_charts/data/charts/aparc.female_normative_trajectories.csv",
    spectral_basis_predictions=spectral_basis_predictions_female,
)
pass

CPU times: user 25 s, sys: 14.9 s, total: 39.9 s
Wall time: 2.94 s


In [16]:
pd.read_csv(
    "/mountpoint/code/projects/normative_brain_charts/data/charts/aparc.female_normative_trajectories.csv",
    index_col=0, header=[0,1],
).shape


(91, 136)

### Destrieux atlas

---

also known as: aparc.a2009s

In [17]:
%%time
compute_normative_trajectories_for_atlas(
    atlas_file="/mountpoint/code/projects/scratch_scripts/scripts_for_niousha/cerbero/Sina_New_Destrieux_atlas.dlabel.nii",
    output_file="/mountpoint/code/projects/normative_brain_charts/data/charts/aparc.a2009s.normative_trajectories.csv",
    spectral_basis_predictions=spectral_basis_predictions,
)
compute_normative_trajectories_for_atlas(
    atlas_file="/mountpoint/code/projects/scratch_scripts/scripts_for_niousha/cerbero/Sina_New_Destrieux_atlas.dlabel.nii",
    output_file="/mountpoint/code/projects/normative_brain_charts/data/charts/aparc.a2009s.male_normative_trajectories.csv",
    spectral_basis_predictions=spectral_basis_predictions_male,
)
compute_normative_trajectories_for_atlas(
    atlas_file="/mountpoint/code/projects/scratch_scripts/scripts_for_niousha/cerbero/Sina_New_Destrieux_atlas.dlabel.nii",
    output_file="/mountpoint/code/projects/normative_brain_charts/data/charts/aparc.a2009s.female_normative_trajectories.csv",
    spectral_basis_predictions=spectral_basis_predictions_female,
)
pass

CPU times: user 10.7 s, sys: 33.1 s, total: 43.8 s
Wall time: 2.86 s


### DKT atlas

---


In [18]:
%%time
compute_normative_trajectories_for_atlas(
    atlas_file="/mountpoint/code/projects/scratch_scripts/scripts_for_niousha/cerbero/Sina_New_DKT_atlas.dlabel.nii",
    output_file="/mountpoint/code/projects/normative_brain_charts/data/charts/aparc.DKTatlas40.normative_trajectories.csv",
    spectral_basis_predictions=spectral_basis_predictions,
)
compute_normative_trajectories_for_atlas(
    atlas_file="/mountpoint/code/projects/scratch_scripts/scripts_for_niousha/cerbero/Sina_New_DKT_atlas.dlabel.nii",
    output_file="/mountpoint/code/projects/normative_brain_charts/data/charts/aparc.DKTatlas40.male_normative_trajectories.csv",
    spectral_basis_predictions=spectral_basis_predictions_male,
)
compute_normative_trajectories_for_atlas(
    atlas_file="/mountpoint/code/projects/scratch_scripts/scripts_for_niousha/cerbero/Sina_New_DKT_atlas.dlabel.nii",
    output_file="/mountpoint/code/projects/normative_brain_charts/data/charts/aparc.DKTatlas40.female_normative_trajectories.csv",
    spectral_basis_predictions=spectral_basis_predictions_female,
)
pass

CPU times: user 4.8 s, sys: 9.22 s, total: 14 s
Wall time: 1.9 s


### Yeo atlas

---


In [19]:
# Yeo 2011 Functional Networks atlas:
# https://github.com/ThomasYeoLab/CBIG/tree/master/stable_projects/brain_parcellation/Yeo2011_fcMRI_clustering/1000subjects_reference/Yeo_JNeurophysiol11_SplitLabels

atlas_names = [
    "Yeo2011_7Networks.split_components",
    "Yeo2011_17Networks.split_components",
    "Yeo2011_7Networks_N1000",
    "Yeo2011_17Networks_N1000",
]

for atlas_name in tqdm(atlas_names):
    compute_normative_trajectories_for_atlas(
        atlas_file=f"/mountpoint/data/templates/YeoAtlas/fs_LR32k/{atlas_name}.dlabel.nii",
        output_file=f"/mountpoint/code/projects/normative_brain_charts/data/charts/{atlas_name}.normative_trajectories.csv",
        spectral_basis_predictions=spectral_basis_predictions, lookup=True,  # map from 64984 vertices to 59412 vertices, excluding the medial wall
    )
    compute_normative_trajectories_for_atlas(
        atlas_file=f"/mountpoint/data/templates/YeoAtlas/fs_LR32k/{atlas_name}.dlabel.nii",
        output_file=f"/mountpoint/code/projects/normative_brain_charts/data/charts/{atlas_name}.male_normative_trajectories.csv",
        spectral_basis_predictions=spectral_basis_predictions_male, lookup=True,  # map from 64984 vertices to 59412 vertices, excluding the medial wall
    )
    compute_normative_trajectories_for_atlas(
        atlas_file=f"/mountpoint/data/templates/YeoAtlas/fs_LR32k/{atlas_name}.dlabel.nii",
        output_file=f"/mountpoint/code/projects/normative_brain_charts/data/charts/{atlas_name}.female_normative_trajectories.csv",
        spectral_basis_predictions=spectral_basis_predictions_female, lookup=True,  # map from 64984 vertices to 59412 vertices, excluding the medial wall
    )

pass


  0%|          | 0/4 [00:00<?, ?it/s]

### HCP Multi-Modal Parcellation (MMP1.0) atlas

---


In [20]:
# HCP MMP1: A Multi-modal Parcellation of Human Cerebral Cortex
# https://balsa.wustl.edu/study/show/RVVG

atlas_name = "HCP_MMP1.0_Glasser"

compute_normative_trajectories_for_atlas(
    atlas_file=f"/mountpoint/data/templates/hcp_parcel/Q1-Q6_RelatedValidation210.CorticalAreas_dil_Final_Final_Areas_Group_Colors.32k_fs_LR.dlabel.nii",
    output_file=f"/mountpoint/code/projects/normative_brain_charts/data/charts/{atlas_name}.normative_trajectories.csv",
    spectral_basis_predictions=spectral_basis_predictions,
)
compute_normative_trajectories_for_atlas(
    atlas_file=f"/mountpoint/data/templates/hcp_parcel/Q1-Q6_RelatedValidation210.CorticalAreas_dil_Final_Final_Areas_Group_Colors.32k_fs_LR.dlabel.nii",
    output_file=f"/mountpoint/code/projects/normative_brain_charts/data/charts/{atlas_name}.male_normative_trajectories.csv",
    spectral_basis_predictions=spectral_basis_predictions_male,
)
compute_normative_trajectories_for_atlas(
    atlas_file=f"/mountpoint/data/templates/hcp_parcel/Q1-Q6_RelatedValidation210.CorticalAreas_dil_Final_Final_Areas_Group_Colors.32k_fs_LR.dlabel.nii",
    output_file=f"/mountpoint/code/projects/normative_brain_charts/data/charts/{atlas_name}.female_normative_trajectories.csv",
    spectral_basis_predictions=spectral_basis_predictions_female,
)

pass


### Gordon atlas

---


In [21]:
# Gordon et al. (2016)
# https://balsa.wustl.edu/WK71

atlas_name = "Gordon333"

compute_normative_trajectories_for_atlas(
    atlas_file=f"/mountpoint/data/templates/hcp_parcel/Gordon333.32k_fs_LR.dlabel.nii",
    output_file=f"/mountpoint/code/projects/normative_brain_charts/data/charts/{atlas_name}.normative_trajectories.csv",
    spectral_basis_predictions=spectral_basis_predictions,
)
compute_normative_trajectories_for_atlas(
    atlas_file=f"/mountpoint/data/templates/hcp_parcel/Gordon333.32k_fs_LR.dlabel.nii",
    output_file=f"/mountpoint/code/projects/normative_brain_charts/data/charts/{atlas_name}.male_normative_trajectories.csv",
    spectral_basis_predictions=spectral_basis_predictions_male,
)
compute_normative_trajectories_for_atlas(
    atlas_file=f"/mountpoint/data/templates/hcp_parcel/Gordon333.32k_fs_LR.dlabel.nii",
    output_file=f"/mountpoint/code/projects/normative_brain_charts/data/charts/{atlas_name}.female_normative_trajectories.csv",
    spectral_basis_predictions=spectral_basis_predictions_female,
)

pass


### Yan atlas

---

**All variations**


In [22]:
# Yan 2023 homotopic parcellation:
# https://github.com/ThomasYeoLab/Standalone_Yan2023_homotopic/tree/master/stable_projects/brain_parcellations/Yan2023_homotopic

atlas_names = [x[:x.index('.')] for x in os.listdir("/mountpoint/data/templates/Yan2023_homotopic/") if 'dlabel.nii' in x]

for atlas_name in tqdm(atlas_names):
    compute_normative_trajectories_for_atlas(
        atlas_file=f"/mountpoint/data/templates/Yan2023_homotopic/{atlas_name}.dlabel.nii",
        output_file=f"/mountpoint/code/projects/normative_brain_charts/data/charts/Yan2023_{atlas_name}.normative_trajectories.csv",
        spectral_basis_predictions=spectral_basis_predictions, lookup=True,  # map from 64984 vertices to 59412 vertices, excluding the medial wall
    )
    compute_normative_trajectories_for_atlas(
        atlas_file=f"/mountpoint/data/templates/Yan2023_homotopic/{atlas_name}.dlabel.nii",
        output_file=f"/mountpoint/code/projects/normative_brain_charts/data/charts/Yan2023_{atlas_name}.male_normative_trajectories.csv",
        spectral_basis_predictions=spectral_basis_predictions_male, lookup=True,  # map from 64984 vertices to 59412 vertices, excluding the medial wall
    )
    compute_normative_trajectories_for_atlas(
        atlas_file=f"/mountpoint/data/templates/Yan2023_homotopic/{atlas_name}.dlabel.nii",
        output_file=f"/mountpoint/code/projects/normative_brain_charts/data/charts/Yan2023_{atlas_name}.female_normative_trajectories.csv",
        spectral_basis_predictions=spectral_basis_predictions_female, lookup=True,  # map from 64984 vertices to 59412 vertices, excluding the medial wall
    )

pass

  0%|          | 0/20 [00:00<?, ?it/s]

### Schaefer atlas

---

**All variations**


In [23]:
# Schaefer 2018 local-global parcellation:
# https://github.com/ThomasYeoLab/CBIG/tree/master/stable_projects/brain_parcellation/Schaefer2018_LocalGlobal

atlas_names = [x[:x.index('.')] for x in os.listdir("/mountpoint/data/templates/Schaefer2018/") if 'dlabel.nii' in x]

for atlas_name in tqdm(atlas_names):
    compute_normative_trajectories_for_atlas(
        atlas_file=f"/mountpoint/data/templates/Schaefer2018/{atlas_name}.dlabel.nii",
        output_file=f"/mountpoint/code/projects/normative_brain_charts/data/charts/{atlas_name}.normative_trajectories.csv",
        spectral_basis_predictions=spectral_basis_predictions, lookup=True,  # map from 64984 vertices to 59412 vertices, excluding the medial wall
    )
    compute_normative_trajectories_for_atlas(
        atlas_file=f"/mountpoint/data/templates/Schaefer2018/{atlas_name}.dlabel.nii",
        output_file=f"/mountpoint/code/projects/normative_brain_charts/data/charts/{atlas_name}.male_normative_trajectories.csv",
        spectral_basis_predictions=spectral_basis_predictions_male, lookup=True,  # map from 64984 vertices to 59412 vertices, excluding the medial wall
    )
    compute_normative_trajectories_for_atlas(
        atlas_file=f"/mountpoint/data/templates/Schaefer2018/{atlas_name}.dlabel.nii",
        output_file=f"/mountpoint/code/projects/normative_brain_charts/data/charts/{atlas_name}.female_normative_trajectories.csv",
        spectral_basis_predictions=spectral_basis_predictions_female, lookup=True,  # map from 64984 vertices to 59412 vertices, excluding the medial wall
    )

pass

  0%|          | 0/20 [00:00<?, ?it/s]

### Von Economo – Koskinas atlas

---


In [24]:
# Scholtens et al. (2018)
# http://www.dutchconnectomelab.nl/economo/

atlas_name = "Economo"

compute_normative_trajectories_for_atlas(
    atlas_file=f"/mountpoint/code/projects/scratch_scripts/scripts_for_niousha/cerbero/Sina_New_Economo_atlas.dlabel.nii",
    output_file=f"/mountpoint/code/projects/normative_brain_charts/data/charts/{atlas_name}.normative_trajectories.csv",
    spectral_basis_predictions=spectral_basis_predictions,
)
compute_normative_trajectories_for_atlas(
    atlas_file=f"/mountpoint/code/projects/scratch_scripts/scripts_for_niousha/cerbero/Sina_New_Economo_atlas.dlabel.nii",
    output_file=f"/mountpoint/code/projects/normative_brain_charts/data/charts/{atlas_name}.male_normative_trajectories.csv",
    spectral_basis_predictions=spectral_basis_predictions_male,
)
compute_normative_trajectories_for_atlas(
    atlas_file=f"/mountpoint/code/projects/scratch_scripts/scripts_for_niousha/cerbero/Sina_New_Economo_atlas.dlabel.nii",
    output_file=f"/mountpoint/code/projects/normative_brain_charts/data/charts/{atlas_name}.female_normative_trajectories.csv",
    spectral_basis_predictions=spectral_basis_predictions_female,
)

pass


### Stats

---



In [25]:
atlas_files = [x for x in os.listdir("/mountpoint/code/projects/normative_brain_charts/data/charts/") if 'normative_trajectories.csv' in x and 'male' not in x]
region_count = sum([pd.read_csv(f"/mountpoint/code/projects/normative_brain_charts/data/charts/{x}", index_col=0, header=[0,1],).shape[1]/2 for x in atlas_files])
print(f"Estimated normative ranges for a total of {len(atlas_files)} parcellation schemes and {region_count} regions")


Estimated normative ranges for a total of 50 parcellation schemes and 23242.0 regions
